In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

df = sns.load_dataset('titanic')
print(df.shape)
df.isna().sum()

(891, 15)


survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

In [2]:
missing_share = df.isna().mean()
high_missing_cols = missing_share[missing_share > 0.5].index.tolist()

target = 'survived'
redundant_cols = ['alive', 'class', 'embark_town', 'adult_male', 'who', 'alone']

cols_to_drop = list(set(high_missing_cols + redundant_cols))
df_clean = df.drop(columns=cols_to_drop)

print('Dropped columns and reasons:')
for col in cols_to_drop:
    if col in high_missing_cols:
        reason = (f"more than 50% of values are missing "
                  f"({missing_share[col]:.0%}), so imputing it would mostly be guessing")
    elif col == 'alive':
        reason = 'a string mirror of the target column (survived); keeping it would leak the label'
    elif col == 'class':
        reason = 'duplicates the information already in pclass (same categories, different dtype)'
    elif col == 'embark_town':
        reason = 'duplicates embarked (same information, full name vs. letter code)'
    elif col == 'adult_male':
        reason = 'a near-duplicate combination of sex and age that adds little beyond those two columns'
    elif col == 'who':
        reason = 'derived from sex and age, so it is largely redundant with those columns'
    elif col == 'alone':
        reason = 'fully determined by sibsp and parch (alone = (sibsp + parch) == 0)'
    else:
        reason = 'low-value column'
    print(f' - {col}: {reason}')

df_clean.head()

Dropped columns and reasons:
 - embark_town: duplicates embarked (same information, full name vs. letter code)
 - class: duplicates the information already in pclass (same categories, different dtype)
 - who: derived from sex and age, so it is largely redundant with those columns
 - alone: fully determined by sibsp and parch (alone = (sibsp + parch) == 0)
 - alive: a string mirror of the target column (survived); keeping it would leak the label
 - adult_male: a near-duplicate combination of sex and age that adds little beyond those two columns
 - deck: more than 50% of values are missing (77%), so imputing it would mostly be guessing


,survived,pclass,sex,age,sibsp,parch,fare,embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [3]:
print('Remaining missing values:')
print(df_clean.isna().sum())

# Strategy 1: median imputation for the numeric 'age' column.
# Median is used instead of mean because age is right-skewed and the median is
# less sensitive to the few very old passengers.
age_median = df_clean['age'].median()
df_clean['age'] = df_clean['age'].fillna(age_median)

# Strategy 2: most-frequent (mode) imputation for the categorical 'embarked' column.
# Only 2 rows are missing, and filling a category with its most common value
# is the standard, low-risk choice for a low-missingness categorical feature.
embarked_mode = df_clean['embarked'].mode()[0]
df_clean['embarked'] = df_clean['embarked'].fillna(embarked_mode)

print('\nMissing values after imputation:')
print(df_clean.isna().sum())

Remaining missing values:
survived      0
pclass        0
sex           0
age         177
sibsp         0
parch         0
fare          0
embarked      2
dtype: int64

Missing values after imputation:
survived    0
pclass      0
sex         0
age         0
sibsp       0
parch       0
fare        0
embarked    0
dtype: int64


In [4]:
categorical_cols = df_clean.select_dtypes(include=['object', 'category', 'bool', 'str']).columns.tolist()
categorical_cols = [c for c in categorical_cols if c != target]
print('Categorical columns to encode:', categorical_cols)

# 'sex' has 2 categories with no natural order, so a single binary column
# (0/1) via one-hot encoding with drop='if_binary' is the most compact choice.
# 'embarked' has 3 categories with no natural order, so it gets one-hot encoded
# into separate indicator columns (no ordinal relationship to preserve).
encoder = OneHotEncoder(drop='if_binary', sparse_output=False, handle_unknown='ignore')
encoded = encoder.fit_transform(df_clean[categorical_cols])
encoded_cols = encoder.get_feature_names_out(categorical_cols)
encoded_df = pd.DataFrame(encoded, columns=encoded_cols, index=df_clean.index)

df_encoded = pd.concat([df_clean.drop(columns=categorical_cols), encoded_df], axis=1)
df_encoded.head()

Categorical columns to encode: ['sex', 'embarked']


,survived,pclass,age,sibsp,parch,fare,sex_male,embarked_C,embarked_Q,embarked_S
0,0,3,22.0,1,0,7.2500,1.0,0.0,0.0,1.0
1,1,1,38.0,1,0,71.2833,0.0,1.0,0.0,0.0
2,1,3,26.0,0,0,7.9250,0.0,0.0,0.0,1.0
3,1,1,35.0,1,0,53.1000,0.0,0.0,0.0,1.0
4,0,3,35.0,0,0,8.0500,1.0,0.0,0.0,1.0


In [5]:
# Numeric columns are identified dynamically (everything except the target and
# the encoded indicator columns, which are already 0/1 and do not need scaling).
numeric_cols = [c for c in ['pclass', 'age', 'sibsp', 'parch', 'fare'] if c in df_encoded.columns]
print('Numeric columns to scale:', numeric_cols)

# StandardScaler is used because these numeric columns are on very different scales
# (e.g. fare ranges into the hundreds while pclass is 1-3), and several downstream
# models (e.g. KNN, SVM, logistic regression) are sensitive to that difference.
scaler = StandardScaler()
df_encoded[numeric_cols] = scaler.fit_transform(df_encoded[numeric_cols])
df_encoded[numeric_cols].describe().loc[['mean', 'std']]

Numeric columns to scale: ['pclass', 'age', 'sibsp', 'parch', 'fare']


,pclass,age,sibsp,parch,fare
mean,-8.772133e-17,2.272780e-16,4.386066e-17,5.382900e-17,3.987333e-18
std,1.000562e+00,1.000562e+00,1.000562e+00,1.000562e+00,1.000562e+00
